In [2]:
# Autoreload
%load_ext autoreload
%autoreload 2

from optimizer import Optimizer
from chromosome import Chromosome
from character import Character
from utils import get_item_set, elements
import json

with open('data/MAPPED_ITEMS.json', 'r') as file:
    items = {i["ankama_id"]: i for i in json.load(file)}

with open('data/MAPPED_SETS.json', 'r') as file:
    item_sets = {i["ankama_id"]: i for i in json.load(file)}

In [3]:
config = {
            "optimizer" : {
                "language": "es",
                "population_size": 300,
                "num_parents_mating": 100, # Number of parents to mate, usually 1/3 of population size
                "num_generations": 400,
                "mutation_rate": 0.05,
                "crossover_rate": 0.8,
                "parent_selection_type": "tournament",  # Tournament Selection
                "tournament_size": 3,
                "exclusions": {
                    "items": [6894,
                              6895,
                              3080,
                              9031,
                              6980 # Vulbis
                              ]  # Excluded items
                },
                "inclusions": {
                    "items": [
                        14093, # Botas kuko
                        18718, # Escudo kuko
                        # 19985, # Anillo de vuelo

                    ]
                },
                "preferences": {
                    "lower": {
                        12: {
                            "target": 12,  # AP
                            "strength": 0.9,  # Penalty for not meeting the target
                        },
                        8: {
                            "target": 6,  # MP
                            "strength": 0.9,  # Penalty for not meeting the target
                        },
                        29: {
                            "target": 80,  # Crit
                            "strength": 0.75,  # Penalty for not meeting the target
                        },
                        9: {
                            "target": 4000,  # Vit
                            "strength": 0.5,  # Penalty for not meeting the target
                        },
                        34 : {
                            "target": 15, # %res neutral
                            "strength": 0.1,  # Penalty for not meeting the target
                        },
                        37 : {
                            "target": 15, # %res fire
                            "strength": 0.1,  # Penalty for not meeting the target
                        },
                        16 : {
                            "target": 15, # %res air
                            "strength": 0.1,  # Penalty for not meeting the target
                        },
                        17 : {
                            "target": 15, # %res water
                            "strength": 0.1,  # Penalty for not meeting the target
                        },
                        63 : {
                            "target": 15, # %res range
                            "strength": 0.1,  # Penalty for not meeting the target
                        }
                    },
                    "upper": {}
                },
                "elements" : [
                    36, # Agility,
                    22, # Chance
                    13, # Intelligence
                    45  # Strength
                ],
                "level_offset": 10,  # Level offset for items
                "objective": "weapon"  # Objective to optimize, can be "weapon" or "elements"
            },
            "character": {
                "level": 200,
                "scrolled": True,
                "exo": {
                    12:1, # Exo AP
                    8:1
                    },  # Exo MP
                "distributed_points": {
                    9: 995,  # Vitality
                }
            },
        }

In [4]:
import pandas as pd

effect_descriptions = {}
for _,item in items.items():
    if not item["effects"]:
        continue
    for effect in item["effects"]:
        effect_descriptions[effect["element_id"]] = effect["type"]

dct = {}
for k,v in items.items():
    item_set = get_item_set(v,item_sets)
    dct[k] = {
        "name" : v["name"][config["optimizer"]["language"]],
        "set" : item_set["name"][config["optimizer"]["language"]] if item_set else None,
        "level" : v["level"]
    }
Items = pd.DataFrame.from_dict(dct, orient='index')

In [5]:
# Items[Items["name"].str.contains("vuelo")]

In [6]:
char = Character(**config["character"])
opt = Optimizer(char,
                 items,
                   item_sets,
                     **config["optimizer"])

Total items: 2748
Total item sets: 148
Total pools:
   16 with sizes [69, 118, 71, 71, 62, 1, 1, 88, 69, 412, 326, 326, 326, 326, 326, 326]
Total combinations: 10^30.88


In [6]:
solutions = opt.optimize(islands = 10, max_workers = 4)

In [14]:
for idx, solution in enumerate(sorted(solutions, key=lambda x: x.fitness, reverse=True)):
    if solution.penalized:
        continue
    print(f"Solution {idx}:")
    print("\tFitness:", solution.get_fitness())
    print("\tWeapon damage:", solution.get_final_weapon_damage())
    print("\tElemental damage:", solution.get_final_elemental_damage())
    # print("\tIs the solution viable?", solution.viable)
    # print("\tWas the solution penalized?", solution.penalized)

    print("\tPreferences:")
    df = solution.totals_summary(effect_descriptions)
    for bound, preference in solution.preferences.items():
        for element, details in preference.items():
            row = df.loc[element]
            print(f"\t\t{row['description']} : {row['value']}")
    print("\tCharacteristics:")
    for element in elements:
        print(f"\t\t{df.loc[element]['description']}: {df.loc[element]['value']}")

    print("\tItems:")
    print("\t\tType\tDescription\tLevel")
    for item,row in solution.set_summary().sort_values("type").iterrows():
        # Prettify the output
        print(f"\t\t{row['type']}\t{row['description']}\t{row['level']}")

Solution 0:
	Fitness: 2152.49
	Weapon damage: 2152.49
	Elemental damage: 1671.25
	Preferences:
		PA : 12.0
		PM : 6.0
		% crítico : 89.0
		vitalidad : 4410.0
		% resistencia neutral : 15.0
		% resistencia al fuego : 21.0
		% resistencia al aire : 45.0
		% resistencia al agua : 41.0
		% resistencia a la tierra : 21.0
	Characteristics:
		agilidad: 530.0
		suerte: 385.0
		inteligencia: 690.0
		fuerza: 540.0
		vitalidad: 4410.0
		sabiduría: 430.0
	Items:
		Type	Description	Level
		Amuleto	Talismán Tira	200
		Anillo	Anillo de Noai Aludem	197
		Anillo	Anillo de kukoloide	200
		Bota	Botas de kukoloide	200
		Capa	Capa Ordiosera	200
		Cinturón	Cinto Dovencible	200
		Dofus	Dofus Púrpura	110
		Dofus	Dofus Turquesa	160
		Dofus	Dofus de los hielos	180
		Dofus	Dofus Ocre	160
		Dofus	Dolmanax	100
		Escudo	Escudo de kukoloide	200
		Espada	Espada maldita del gran Desangrador Guerrero	200
		Mascota	Lokulto	20
		Sombrero	Gorro de Noai Aludem	198
		Trofeo	Suertudo menor	50
Solution 1:
	Fitness: 1766.76
	W

In [8]:
solution.totals_summary(effect_descriptions)#.sort_values("value",ascending=False)

,description,value
0,Intercambiable:,0.0
8,PM,6.0
9,vitalidad,4945.0
10,sabiduría,415.0
12,PA,11.0
13,inteligencia,420.0
15,de resistencia a la tierra,15.0
16,% resistencia al aire,36.0
17,% resistencia al agua,25.0
22,suerte,400.0


In [9]:
set_summary = solution.set_summary()
set_summary

,type_id,type,description,level,set
694,13,Dofus,Dofus Púrpura,110,None
739,13,Dofus,Dofus Turquesa,160,None
7043,13,Dofus,Dofus de los hielos,180,None
11738,3,Anillo,Anillo Chanicomo,194,None
13344,13,Dofus,Dolmanax,100,None
13673,12,Mascota,Lokulto,20,None
13759,13,Trofeo,Hércules mayor,150,None
14093,5,Bota,Botas de kukoloide,200,Set de kukoloide
14168,11,Capa,Capa Pología,198,None
16332,13,Trofeo,Lozano mayor,150,None


In [10]:
set_summary["set"].value_counts()

set
Set de kukoloide           2
Set del Espíritu Insano    2
Set de Cocuzko             1
Set de Wulán               1
Set de los Martilloseco    1
Name: count, dtype: int64

In [11]:
import matplotlib.pyplot as plt
plt.plot(opt.ga_instance.best_solutions_fitness)
plt.xlabel('Generation')
plt.ylabel('Best Fitness')

AttributeError: 'Optimizer' object has no attribute 'ga_instance'

In [ ]:
assert False

AssertionError: 

In [7]:
# Current
chr = Chromosome([
                    8993, # Espada
                    13641, 13643, 13642, # Noai
                    14161, 14162, 14169, # Charlatan
                    19985, # Vuelo
                    18670, # Escudo
                    13673, # Lokulto
                    739, 694, 7754, 737, 7043, 17078 # Dofus
                 ],
                 opt)
chr.fitness

1890.67

In [ ]:
Items[Items["name"].str.contains("kuko")]

,name,set,level
13989,Pluma de kukoloide,None,200
14091,Amuleto de kukoloide,Set de kukoloide,200
14092,Anillo de kukoloide,Set de kukoloide,200
14093,Botas de kukoloide,Set de kukoloide,200
13988,Ojo de kukoloide,None,200
18718,Escudo de kukoloide,Set de kukoloide,200


In [8]:
# Project
chr = Chromosome(
                 [
                    8993, # Espada
                    13641, 13642, # Noai
                    14161, 14162, 14169, # Charlatan
                    14092, 14093, 18718, # Kuko
                    13673, # Lokulto
                    739, 694, 7754, 737, 7043, 17078 # Dofus
                 ],
                 opt)
chr.fitness

2049.74

In [ ]:
chr.set_summary()

In [ ]:
chr.set_summary()["set"].value_counts()

In [ ]:
chr.totals_summary(effect_descriptions)

In [ ]:
Items[Items["name"].str.contains("Chani")]

In [ ]:
item = items[21392]
# item = items[6981]

In [ ]:
chr.are_item_conditions_met(item)

In [ ]:
item

In [ ]:
Items

In [ ]:
lst

In [ ]:
lst[0][2]

In [ ]:
chr.are_item_conditions_met(items[11738])

In [ ]:
items[11738]["conditions"]

In [ ]:
chr.totals[8]

In [ ]:
lst = []
for i in items.values():
    if i["type"]["superTypeId"] not in [i[0] for i in opt.pool_types]:
        continue
    if i["conditions"]:
        print(opt.solution.are_item_conditions_met(i))
        lst.append((i,opt.solution.are_item_conditions_met(i)))

In [ ]:
lst[1]

In [ ]:
lst.count(True), lst.count(False)